In [13]:
"""
This notebook is for taking FITS files saved from the evaluate cli and analyzing the estimator's performance.
"""
import os
os.environ['XLA_FLAGS'] = '--xla_gpu_strict_conv_algorithm_picker=false'

import sys
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import subprocess
import warnings
warnings.filterwarnings('ignore')

import jax
jax.clear_caches()

sys.path.append('/home/adfield/ShearNet/')
from shearnet.cli.evaluate import load_config, create_parser, initialize_model

# ANSI color codes
BOLD = '\033[1m'
YELLOW = '\033[93m'
CYAN = '\033[96m'
GREEN = '\033[92m'
RED = '\033[91m'
END = '\033[0m'

# GPU diagnostics
print(f"JAX devices: {jax.devices()}")
print(f"Default backend: {jax.default_backend()}")
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(result.stdout[:500])
except:
    print("nvidia-smi not available")

# Input your FITS file path here
path = '/home/adfield/ShearNet/plots/first_validation_test_2_params/evaluation_data.fits'

JAX devices: [CudaDevice(id=0)]
Default backend: gpu
Mon Jan 26 19:45:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.133.20             Driver Version: 570.133.20     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|       


In [14]:
def extract_info_from_fits(filepath):
    """
    Returns a dictionary of info from FITS file containing shear estimation results.
    
    Returns:
        dict: Dictionary containing TRUE, NN (ShearNet), and NGMIX parameters
              (g1, g2, sigma, flux) for comparison
    """
    with fits.open(filepath) as hdul:
        TRUE_G1 = hdul[1].data['TRUE_G1']
        TRUE_G2 = hdul[1].data['TRUE_G2']
        TRUE_SIGMA = hdul[1].data['TRUE_SIGMA']
        TRUE_FLUX = hdul[1].data['TRUE_FLUX']
        NN_G1 = hdul[1].data['NN_G1']
        NN_G2 = hdul[1].data['NN_G2']
        NN_SIGMA = hdul[1].data['NN_SIGMA']
        NN_FLUX = hdul[1].data['NN_FLUX']
        NGMIX_G1 = hdul[1].data['NGMIX_G1']
        NGMIX_G2 = hdul[1].data['NGMIX_G2']
        NGMIX_SIGMA = hdul[1].data['NGMIX_SIGMA']
        NGMIX_FLUX = hdul[1].data['NGMIX_FLUX']
        CLEAN_IMAGES = hdul[2].data
        GALAXY_IMAGES = hdul[3].data
        PSF_IMAGES = hdul[4].data
        NOISE_IMAGES = hdul[5].data
        WEIGHT_MAPS = hdul[6].data
        
    return {
        'TRUE_G1': TRUE_G1,
        'TRUE_G2': TRUE_G2,
        'TRUE_SIGMA': TRUE_SIGMA,
        'TRUE_FLUX': TRUE_FLUX,
        'NN_G1': NN_G1,
        'NN_G2': NN_G2,
        'NN_SIGMA': NN_SIGMA,
        'NN_FLUX': NN_FLUX,
        'NGMIX_G1': NGMIX_G1,
        'NGMIX_G2': NGMIX_G2,
        'NGMIX_SIGMA': NGMIX_SIGMA,
        'NGMIX_FLUX': NGMIX_FLUX,
        'CLEAN_IMAGES': CLEAN_IMAGES,
        'GALAXY_IMAGES': GALAXY_IMAGES,
        'PSF_IMAGES': PSF_IMAGES,
        'NOISE_IMAGES': NOISE_IMAGES,
        'WEIGHT_MAPS': WEIGHT_MAPS
    }

In [15]:
def calculate_multiplicative_bias_from_single_fits(state, fits_data, 
                                                    true_shear_step=0.02, 
                                                    batch_size=32, 
                                                    h=0.01,
                                                    pixel_scale=0.141,
                                                    model_type='fork',  # Changed default
                                                    psf_images=None,
                                                    Njack=20):
    """
    Calculate multiplicative (m) and additive (c) bias from a SINGLE FITS file.
    
    This function generates all sheared versions (±0.02 for g1 and g2) from the 
    CLEAN_IMAGES in the FITS file, then calculates response matrices and bias.
    """
    import galsim
    import jax.numpy as jnp
    
    print(f"\n{BOLD}{'='*70}")
    print("CALCULATING MULTIPLICATIVE BIAS FROM SINGLE FITS FILE")
    print(f"{'='*70}{END}")
    print(f"Shear to apply: ±{true_shear_step}")
    print(f"Response perturbation: ±{h}")
    print(f"Pixel scale: {pixel_scale} arcsec/pixel")
    print(f"Jackknife samples: {Njack}")
    print(f"Model type: {model_type}")
    print(f"Number of galaxies: {len(fits_data['CLEAN_IMAGES'])}")
    
    clean_images = fits_data['CLEAN_IMAGES']
    psf_imgs = fits_data['PSF_IMAGES'] if psf_images is None else psf_images
    noise_images = fits_data['NOISE_IMAGES']
    n_gal = len(clean_images)
    npix = clean_images.shape[1]
    
    gsp = galsim.GSParams(maximum_fft_size=32768)
    
    def apply_shear_and_reconvolve(g1_shear, g2_shear):
        """
        Apply shear to clean images, reconvolve with PSF, add noise.
        Returns galaxy images ready for model prediction.
        """
        print(f"  Generating dataset with g1={g1_shear:+.3f}, g2={g2_shear:+.3f}...")
        sheared_images = np.zeros_like(clean_images)
        
        for i in range(n_gal):
            clean_gal_image = galsim.Image(clean_images[i], scale=pixel_scale)
            clean_gal = galsim.InterpolatedImage(clean_gal_image)
            
            psf_image = galsim.Image(psf_imgs[i], scale=pixel_scale)
            psf = galsim.InterpolatedImage(psf_image)
            
            # Apply shear
            sheared_gal = clean_gal.shear(g1=g1_shear, g2=g2_shear)
            
            # Convolve with PSF
            convolved = galsim.Convolve([sheared_gal, psf], gsparams=gsp)
            
            # Draw and add noise
            convolved_im = convolved.drawImage(nx=npix, ny=npix, scale=pixel_scale).array
            sheared_images[i] = convolved_im + noise_images[i]
        
        return sheared_images
    
    def get_predictions(images, psf_imgs_input=None):
        """Get model predictions for a set of images."""
        preds = []
        for i in range(0, n_gal, batch_size):
            batch_slice = slice(i, min(i + batch_size, n_gal))
            if model_type == 'fork':
                if psf_imgs_input is None:
                    psf_imgs_input = psf_imgs
                batch_pred = state.apply_fn(
                    state.params,
                    images[batch_slice],
                    psf_imgs_input[batch_slice],
                    deterministic=True
                )
            else:
                batch_pred = state.apply_fn(
                    state.params,
                    images[batch_slice],
                    deterministic=True
                )
            preds.append(batch_pred)
        return jnp.concatenate(preds)
    
    def jackknife_estimate(values):
        """Calculate jackknife mean and error."""
        n = len(values)
        indices = np.arange(n)
        chunks = np.array_split(indices, Njack)
        
        jack_values = []
        for chunk in chunks:
            mask_j = np.ones(n, dtype=bool)
            mask_j[chunk] = False
            jack_values.append(np.mean(values[mask_j]))
        
        jack_values = np.array(jack_values)
        mean = np.mean(jack_values)
        err = np.sqrt((Njack - 1) / Njack * np.sum((jack_values - mean)**2))
        
        return mean, err
    
    # ========== Step 1: Calculate Response Matrix ONCE (at zero shear) ==========
    print(f"\n{BOLD}{CYAN}Calculating Response Matrix (once)...{END}")
    
    # Generate response-sheared images at zero base shear
    print("  Generating +h g1 images...")
    e1_pos = apply_shear_and_reconvolve(+h, 0.0)
    print("  Generating -h g1 images...")
    e1_neg = apply_shear_and_reconvolve(-h, 0.0)
    print("  Generating +h g2 images...")
    e2_pos = apply_shear_and_reconvolve(0.0, +h)
    print("  Generating -h g2 images...")
    e2_neg = apply_shear_and_reconvolve(0.0, -h)
    
    # Get predictions for response calculation
    preds_e1p = get_predictions(e1_pos, psf_imgs)
    preds_e1m = get_predictions(e1_neg, psf_imgs)
    preds_e2p = get_predictions(e2_pos, psf_imgs)
    preds_e2m = get_predictions(e2_neg, psf_imgs)
    
    # Calculate per-galaxy response components
    R_11 = np.array((preds_e1p[:, 0] - preds_e1m[:, 0]) / (2 * h))
    R_12 = np.array((preds_e2p[:, 0] - preds_e2m[:, 0]) / (2 * h))
    R_21 = np.array((preds_e1p[:, 1] - preds_e1m[:, 1]) / (2 * h))
    R_22 = np.array((preds_e2p[:, 1] - preds_e2m[:, 1]) / (2 * h))
    
    # Mean response matrix
    R11_mean, R11_err = jackknife_estimate(R_11)
    R12_mean, R12_err = jackknife_estimate(R_12)
    R21_mean, R21_err = jackknife_estimate(R_21)
    R22_mean, R22_err = jackknife_estimate(R_22)
    
    print(f"\n{CYAN}Response Matrix:{END}")
    print(f"  R₁₁ = {R11_mean:.6f} ± {R11_err:.6f}")
    print(f"  R₁₂ = {R12_mean:.6f} ± {R12_err:.6f}")
    print(f"  R₂₁ = {R21_mean:.6f} ± {R21_err:.6f}")
    print(f"  R₂₂ = {R22_mean:.6f} ± {R22_err:.6f}")
    
    # ========== Step 2: Generate base-sheared datasets ==========
    print(f"\n{BOLD}{CYAN}Generating base-sheared datasets...{END}")
    
    images_g1_pos = apply_shear_and_reconvolve(+true_shear_step, 0.0)
    images_g1_neg = apply_shear_and_reconvolve(-true_shear_step, 0.0)
    images_g2_pos = apply_shear_and_reconvolve(0.0, +true_shear_step)
    images_g2_neg = apply_shear_and_reconvolve(0.0, -true_shear_step)
    
    # ========== Step 3: Get ellipticity predictions ==========
    print(f"\n{BOLD}{CYAN}Getting ellipticity predictions...{END}")
    
    preds_g1_pos = get_predictions(images_g1_pos, psf_imgs)
    preds_g1_neg = get_predictions(images_g1_neg, psf_imgs)
    preds_g2_pos = get_predictions(images_g2_pos, psf_imgs)
    preds_g2_neg = get_predictions(images_g2_neg, psf_imgs)
    
    e1_from_g1_pos = np.array(preds_g1_pos[:, 0])
    e1_from_g1_neg = np.array(preds_g1_neg[:, 0])
    e2_from_g2_pos = np.array(preds_g2_pos[:, 1])
    e2_from_g2_neg = np.array(preds_g2_neg[:, 1])
    
    # ========== Step 4: Calculate bias with jackknife ==========
    print(f"\n{BOLD}{CYAN}Calculating bias with jackknife...{END}")
    
    # Filter NaNs for g1
    mask_g1 = np.isfinite(R_11) & np.isfinite(e1_from_g1_pos) & np.isfinite(e1_from_g1_neg)
    R11_filt = R_11[mask_g1]
    e1_pos_filt = e1_from_g1_pos[mask_g1]
    e1_neg_filt = e1_from_g1_neg[mask_g1]
    print(f"  Valid galaxies for g1: {np.sum(mask_g1)}/{len(mask_g1)}")
    
    # Filter NaNs for g2
    mask_g2 = np.isfinite(R_22) & np.isfinite(e2_from_g2_pos) & np.isfinite(e2_from_g2_neg)
    R22_filt = R_22[mask_g2]
    e2_pos_filt = e2_from_g2_pos[mask_g2]
    e2_neg_filt = e2_from_g2_neg[mask_g2]
    print(f"  Valid galaxies for g2: {np.sum(mask_g2)}/{len(mask_g2)}")
    
    # Per-galaxy gamma estimates
    gamma1_per = (e1_pos_filt - e1_neg_filt) / (2.0 * R11_filt)
    gamma2_per = (e2_pos_filt - e2_neg_filt) / (2.0 * R22_filt)
    
    # Jackknife for m1
    indices_g1 = np.arange(len(gamma1_per))
    chunks_g1 = np.array_split(indices_g1, Njack)
    m1_jack = []
    for chunk in chunks_g1:
        mask_j = np.ones(len(gamma1_per), dtype=bool)
        mask_j[chunk] = False
        gamma1_est_jack = np.mean(gamma1_per[mask_j])
        m1_jack.append(gamma1_est_jack / true_shear_step - 1)
    
    m1_jack = np.array(m1_jack)
    m1 = np.mean(m1_jack)
    m1_err = np.sqrt((Njack - 1) / Njack * np.sum((m1_jack - m1)**2))
    
    # Jackknife for m2
    indices_g2 = np.arange(len(gamma2_per))
    chunks_g2 = np.array_split(indices_g2, Njack)
    m2_jack = []
    for chunk in chunks_g2:
        mask_j = np.ones(len(gamma2_per), dtype=bool)
        mask_j[chunk] = False
        gamma2_est_jack = np.mean(gamma2_per[mask_j])
        m2_jack.append(gamma2_est_jack / true_shear_step - 1)
    
    m2_jack = np.array(m2_jack)
    m2 = np.mean(m2_jack)
    m2_err = np.sqrt((Njack - 1) / Njack * np.sum((m2_jack - m2)**2))
    
    # Additive bias c1, c2
    gamma_est_g1_pos_per = e1_pos_filt / R11_filt
    gamma_est_g1_neg_per = e1_neg_filt / R11_filt
    c1_per = 0.5 * (gamma_est_g1_pos_per + gamma_est_g1_neg_per)
    c1, c1_err = jackknife_estimate(c1_per)
    
    gamma_est_g2_pos_per = e2_pos_filt / R22_filt
    gamma_est_g2_neg_per = e2_neg_filt / R22_filt
    c2_per = 0.5 * (gamma_est_g2_pos_per + gamma_est_g2_neg_per)
    c2, c2_err = jackknife_estimate(c2_per)
    
    # ========== Final Results ==========
    print(f"\n{BOLD}{'='*70}")
    print("BIAS CALIBRATION RESULTS WITH JACKKNIFE ERRORS")
    print(f"{'='*70}{END}")
    print(f"\n{BOLD}{YELLOW}Component 1:{END}")
    print(f"{BOLD}  m₁ = {m1:+.6f} ± {m1_err:.6f}  ({m1*100:+.2f}% ± {m1_err*100:.2f}%){END}")
    print(f"{BOLD}  c₁ = {c1:+.6f} ± {c1_err:.6f}{END}")
    print(f"\n{BOLD}{YELLOW}Component 2:{END}")
    print(f"{BOLD}  m₂ = {m2:+.6f} ± {m2_err:.6f}  ({m2*100:+.2f}% ± {m2_err*100:.2f}%){END}")
    print(f"{BOLD}  c₂ = {c2:+.6f} ± {c2_err:.6f}{END}")
    print(f"{BOLD}{'='*70}{END}\n")
    
    return {
        'm1': m1, 'm1_err': m1_err,
        'c1': c1, 'c1_err': c1_err,
        'm2': m2, 'm2_err': m2_err,
        'c2': c2, 'c2_err': c2_err,
        'R11': R11_mean, 'R11_err': R11_err,
        'R22': R22_mean, 'R22_err': R22_err,
    }

In [16]:
def initialize_model_from_fits(fits_path, config):
    """
    Initialize model using the architecture.py saved alongside the FITS file.
    
    Args:
        fits_path: Path to the FITS file
        config: Config dict from load_config()
    
    Returns:
        state: Loaded model state
    """
    import importlib.util
    import jax.numpy as jnp
    import jax.random as random
    import optax
    from flax.training import checkpoints, train_state
    
    # Get directory containing the FITS file
    model_dir = os.path.dirname(fits_path)
    architecture_path = os.path.join(model_dir, 'architecture.py')
    
    if not os.path.exists(architecture_path):
        raise FileNotFoundError(f"No architecture.py found at {architecture_path}")
    
    print(f"{BOLD}Loading architecture from: {architecture_path}{END}")
    
    # Dynamically load the architecture module
    spec = importlib.util.spec_from_file_location("architecture", architecture_path)
    arch_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(arch_module)
    
    # Load FITS to get image shapes
    with fits.open(fits_path) as hdul:
        galaxy_images = hdul[3].data  # GALAXY_IMAGES
        psf_images = hdul[4].data      # PSF_IMAGES
    
    # Get model class and instantiate
    nn_type = config['nn']
    process_psf = config['process_psf']
    
    if nn_type == "mlp":
        model = arch_module.SimpleGalaxyNN()
    elif nn_type == "cnn":
        model = arch_module.EnhancedGalaxyNN()
    elif nn_type == "resnet":
        model = arch_module.GalaxyResNet()
    elif nn_type == "research_backed":
        model = arch_module.ResearchBackedGalaxyResNet()
    elif nn_type == "forklens_psfnet":
        model = arch_module.ForkLensPSFNet()
    elif nn_type == "fork-like":
        galaxy_type = config.get('galaxy_type', 'research_backed')
        psf_type = config.get('psf_type', 'forklens_psf')
        model = arch_module.ForkLike(galaxy_model_type=galaxy_type, psf_model_type=psf_type)
    else:
        raise ValueError(f"Invalid model type: {nn_type}")
    
    print(f"Model type: {nn_type}")
    
    # Initialize parameters
    rng_key = random.PRNGKey(config['seed'])
    
    if process_psf:
        init_params = model.init(rng_key, jnp.ones_like(galaxy_images[0]), jnp.ones_like(psf_images[0]))
    else:
        init_params = model.init(rng_key, jnp.ones_like(galaxy_images[0]))
    
    state = train_state.TrainState.create(
        apply_fn=model.apply, params=init_params, tx=optax.adam(1e-3)
    )
    
    # Find and load checkpoint
    load_path = os.path.abspath(config['save_path'])
    model_name = config['model_name']
    
    matching_dirs = [
        d for d in os.listdir(load_path) 
        if os.path.isdir(os.path.join(load_path, d)) and d.startswith(model_name)
    ]
    
    if not matching_dirs:
        raise FileNotFoundError(f"No checkpoint directory found in {load_path} starting with '{model_name}'")
    
    # Use most recent checkpoint
    checkpoint_dir = os.path.join(load_path, sorted(matching_dirs)[-1])
    print(f"{GREEN}Loading checkpoint from: {checkpoint_dir}{END}")
    
    state = checkpoints.restore_checkpoint(ckpt_dir=checkpoint_dir, target=state)
    print(f"{GREEN}✓ Model loaded successfully{END}")
    
    return state

In [17]:
fits_data = extract_info_from_fits(path)

model_name = path.split("/")[-2]

parser = create_parser()
args = parser.parse_args(["--model_name", model_name])
config = load_config(args)

# Use the new function that loads from architecture.py
state = initialize_model_from_fits(path, config)

shear_bias = calculate_multiplicative_bias_from_single_fits(state=state, fits_data=fits_data)

print(shear_bias)


Loading model config from: /home/adfield/ShearNet/plots/first_validation_test_2_params/training_config.yaml

Evaluation Configuration

evaluation:
  test_samples: 30000
  seed: 58

model:
  process_psf: True
  type: fork-like
  galaxy: {'type': 'research_backed'}
  psf: {'type': 'forklens_psf'}

plotting:
  plot: True

comparison:
  mcal: True
  ngmix: True
  psf_model: gauss
  gal_model: gauss

Loading architecture from: /home/adfield/ShearNet/plots/first_validation_test_2_params/architecture.py
Model type: fork-like


W0126 19:45:19.871835 1229033 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H200. Using default config.
W0126 19:45:19.898311 1229033 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H200. Using default config.
W0126 19:45:19.937341 1229033 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H200. Using default config.
W0126 19:45:19.957796 1229033 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H200. Using default config.
W0126 19:45:19.999813 1229033 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H200. Using default config.
W0126 19:45:20.020388 1229033 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H200. Using default config.
W0126 19:45:20.040015 1229033 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H200. Using default config.
W0126 19:45:20.137300 1229033 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H200. Using default config.
W0126 19

Loading checkpoint from: /home/adfield/ShearNet/model_checkpoint/first_validation_test_2_params135
✓ Model loaded successfully

CALCULATING MULTIPLICATIVE BIAS FROM SINGLE FITS FILE
Shear to apply: ±0.02
Response perturbation: ±0.01
Pixel scale: 0.141 arcsec/pixel
Jackknife samples: 20
Model type: fork
Number of galaxies: 30000

Calculating Response Matrix (once)...
  Generating +h g1 images...
  Generating dataset with g1=+0.010, g2=+0.000...
  Generating -h g1 images...
  Generating dataset with g1=-0.010, g2=+0.000...
  Generating +h g2 images...
  Generating dataset with g1=+0.000, g2=+0.010...
  Generating -h g2 images...
  Generating dataset with g1=+0.000, g2=-0.010...


W0126 19:50:52.861957 1229033 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H200. Using default config.
W0126 19:50:52.885828 1229033 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H200. Using default config.
W0126 19:50:52.911160 1229033 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H200. Using default config.
W0126 19:50:52.953816 1229033 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H200. Using default config.
W0126 19:50:52.975223 1229033 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H200. Using default config.
W0126 19:50:53.015447 1229033 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H200. Using default config.
W0126 19:50:53.036958 1229033 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H200. Using default config.
W0126 19:50:53.078410 1229033 sol_gpu_cost_model.cc:102] No SoL config found for device: NVIDIA H200. Using default config.
W0126 19


Response Matrix:
  R₁₁ = 0.992517 ± 0.000945
  R₁₂ = -0.002834 ± 0.000906
  R₂₁ = 0.005692 ± 0.001011
  R₂₂ = 0.987158 ± 0.000871

Generating base-sheared datasets...
  Generating dataset with g1=+0.020, g2=+0.000...
  Generating dataset with g1=-0.020, g2=+0.000...
  Generating dataset with g1=+0.000, g2=+0.020...
  Generating dataset with g1=+0.000, g2=-0.020...

Getting ellipticity predictions...

Calculating bias with jackknife...
  Valid galaxies for g1: 30000/30000
  Valid galaxies for g2: 30000/30000

BIAS CALIBRATION RESULTS WITH JACKKNIFE ERRORS

Component 1:
  m₁ = +0.003251 ± 0.000619  (+0.33% ± 0.06%)
  c₁ = +0.000002 ± 0.001970

Component 2:
  m₂ = +0.005163 ± 0.003918  (+0.52% ± 0.39%)
  c₂ = -0.034088 ± 0.040111

{'m1': np.float32(0.0032509088), 'm1_err': np.float32(0.00061885733), 'c1': np.float32(2.198116e-06), 'c1_err': np.float32(0.0019695773), 'm2': np.float32(0.0051633953), 'm2_err': np.float32(0.0039181192), 'c2': np.float32(-0.034087814), 'c2_err': np.float32(0.